## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys
def execute_program():
    input_stream = sys.stdin.buffer.read()
    num_list = list(map(int, input_stream.split()))
    if not num_list:
        return 
    total_size = num_list[0]
    val_x = num_list[1]
    val_y = num_list[2]
    data_array = num_list[3:3 + total_size]
    action_log = []

    def get_lowest_bit(num):
        return num & -num

    shift_val = (val_x - val_y + total_size) % total_size
    block_size = get_lowest_bit(shift_val)
    if block_size == 0:
        block_size = total_size

    def flip_swap():
        action_log.append(0)
        for idx in range(total_size):
            current = data_array[idx]
            if current == val_x:
                data_array[idx] = val_y
            elif current == val_y:
                data_array[idx] = val_x

    def apply_add(step):
        mod_step = step % total_size
        if mod_step == 0:
            return
        action_log.append(mod_step)
        for idx in range(total_size):
            data_array[idx] = (data_array[idx] + mod_step) % total_size

    def apply_xor(step):
        if step == 0:
            return
        action_log.append(-step)
        for idx in range(total_size):
            data_array[idx] ^= step

    def compute_values(a_val, b_val):
        diff = (b_val - a_val + total_size - block_size + total_size) % total_size
        res1 = 0
        res2 = 0
        current_step = total_size // 2

        while current_step >= 2 * block_size:
            if diff >= current_step:
                diff -= current_step
                res2 += current_step // 2
            else:
                res1 += current_step // 2
            current_step = current_step // 2

        res1 += total_size // 2
        res1 += a_val & (block_size - 1)
        res2 += a_val & (block_size - 1)
        return res1, res2

    def exchange_pos(pos1, pos2):
        group1 = (pos1 // block_size) % 2
        group2 = (pos2 // block_size) % 2

        if group1 == group2:
            if group1 == 0:
                mid = (pos1 & (block_size - 1)) + block_size
            else:
                mid = pos1 & (block_size - 1)
            exchange_pos(pos1, mid)
            exchange_pos(pos2, mid)
            exchange_pos(pos1, mid)
        else:
            base1, base2 = compute_values(val_x, val_y)
            curr1, curr2 = compute_values(pos1, pos2)

            apply_add((curr1 - pos1 + total_size) % total_size)
            apply_xor(curr1 ^ base1)
            apply_add((val_x - base1 + total_size) % total_size)
            flip_swap()
            apply_add((base1 - val_x + total_size) % total_size)
            apply_xor(curr1 ^ base1)
            apply_add((pos1 - curr1 + total_size) % total_size)

    def validate_sequence(seq, length):
        if length == 1:
            return True, []
        half_len = length // 2
        left_seq = [seq[i * 2] // 2 for i in range(half_len)]
        right_seq = [seq[i * 2 + 1] // 2 for i in range(half_len)]

        left_valid, left_steps = validate_sequence(left_seq, half_len)
        if not left_valid:
            return False, []
        right_valid, right_steps = validate_sequence(right_seq, half_len)
        if not right_valid:
            return False, []

        final_steps = []
        if seq[0] % 2 == 1:
            if length == 2:
                final_steps.append(1)
            else:
                final_steps.append(-1)

        xor_left = 0
        for act in left_steps:
            if act > 0:
                final_steps.append(-1)
                final_steps.append(1)
            else:
                scaled = act * 2
                final_steps.append(scaled)
                xor_left ^= (-scaled)
        if xor_left != 0:
            final_steps.append(-xor_left)

        xor_right = 0
        for act in right_steps:
            if act > 0:
                final_steps.append(1)
                final_steps.append(-1)
            else:
                scaled = act * 2
                final_steps.append(scaled)
                xor_right ^= (-scaled)

        if (xor_right & half_len) != (xor_left & half_len):
            return False, []
        if xor_left >= half_len:
            xor_left -= half_len
        if xor_right >= half_len:
            xor_right -= half_len
        if xor_left != xor_right:
            return False, []

        compressed = []
        for act in final_steps:
            if not compressed:
                compressed.append(act)
            else:
                if act < 0 and compressed[-1] < 0:
                    combined = -((-compressed[-1]) ^ (-act))
                    if combined != 0:
                        compressed[-1] = combined
                    else:
                        compressed.pop()
                else:
                    compressed.append(act)
        return True, compressed

    if block_size > 1:
        low_part = [data_array[i] & (block_size - 1) for i in range(block_size)]
        valid, op_list = validate_sequence(low_part, block_size)
        if not valid:
            print(-1)
            return
        for cmd in op_list:
            if cmd > 0:
                apply_add(cmd)
            else:
                apply_xor(-cmd)

    for base in range(block_size):
        subgroup = []
        for pos in range(base, total_size, block_size):
            subgroup.append(data_array[pos])
        subgroup.sort()
        ptr = 0
        valid_group = True
        for pos in range(base, total_size, block_size):
            if subgroup[ptr] != pos:
                valid_group = False
                break
            ptr += 1
        if not valid_group:
            print(-1)
            return
        for pos in range(base, total_size, block_size):
            if data_array[pos] != pos:
                exchange_pos(pos, data_array[pos])

    for idx in range(total_size):
        if data_array[idx] != idx:
            print(-1)
            return

    output_lines = [str(len(action_log))]
    for cmd in action_log:
        if cmd == 0:
            output_lines.append("0")
        elif cmd < 0:
            output_lines.append(f"1 {-cmd}")
        else:
            output_lines.append(f"2 {cmd}")
    sys.stdout.write("\n".join(output_lines))

if __name__ == "__main__":
    execute_program()

## B 长跑

In [ ]:
## add your code here
import sys

def main():
    data = list(map(int, sys.stdin.read().split()))
    ptr = 0
    result = []
    
    while ptr < len(data):
        N, L, Maxn, S = data[ptr:ptr+4]
        ptr += 4
        
        cost_map = dict()
        for _ in range(N):
            pos, cost = data[ptr], data[ptr+1]
            ptr += 2
            if pos not in cost_map or cost < cost_map[pos]:
                cost_map[pos] = cost
        
        if Maxn >= L:
            result.append("Yes")
            continue
        
        pos_list = [0]
        cost_list = [0]
        for position in sorted(cost_map):
            if position < L:
                pos_list.append(position)
                cost_list.append(cost_map[position])
        pos_list.append(L)
        cost_list.append(0)
        
        total_nodes = len(pos_list)
        min_cost = [float('inf')] * total_nodes
        min_cost[0] = 0
        
        reachable = False
        for current in range(total_nodes):
            if min_cost[current] > S:
                continue
            next_node = current + 1
            while next_node < total_nodes and pos_list[next_node] - pos_list[current] <= Maxn:
                new_cost = min_cost[current] + cost_list[next_node]
                if pos_list[next_node] == L and min_cost[current] <= S:
                    reachable = True
                    break
                if new_cost < min_cost[next_node]:
                    min_cost[next_node] = new_cost
                next_node += 1
            if reachable:
                break
        
        result.append("Yes" if reachable else "No")
    
    print('\n'.join(result))

if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## add your code here
import sys
def calculate_result():
    raw_input = sys.stdin.read().split()
    if not raw_input:
        return
    
    str_len = int(raw_input[0])
    str_x = raw_input[1]
    str_y = raw_input[2]
    
    if str_len == 0:
        print(0)
        return
    
# 哈希配置：大素数模数 + 基数，防碰撞
    hash_mod = (1 << 61) - 1
    hash_base = 131
    pow_array = [1] * (str_len + 1)
    
    for idx in range(1, str_len + 1):
        pow_array[idx] = (pow_array[idx - 1] * hash_base) % hash_mod

#构建字符串前缀哈希
    def generate_hash(target):
        hash_arr = [0] * (str_len + 1)
        for pos in range(str_len):
            hash_arr[pos + 1] = (hash_arr[pos] * hash_base + ord(target[pos])) % hash_mod
        return hash_arr

#预处理反转字符串哈希,目标字符串哈希
    rev_x = str_x[::-1]
    hash_rev_x = generate_hash(rev_x)
    hash_y = generate_hash(str_y)

    # O(1) 哈希比对函数
    def is_equal(a_start, b_start, match_len):
        if match_len == 0:
            return True
        val1 = (hash_rev_x[a_start + match_len - 1] - hash_rev_x[a_start - 1] * pow_array[match_len]) % hash_mod
        val2 = (hash_y[b_start + match_len - 1] - hash_y[b_start - 1] * pow_array[match_len]) % hash_mod
        return val1 == val2

#Manacher算法：获取回文半径数组
    def manacher_process(s):
        expand_str = ['#'] * (2 * str_len + 1)
        for p in range(str_len):
            expand_str[2 * p + 1] = s[p]
        
        radius = [0] * (2 * str_len + 1)
        center = 0
        right = 0
        
        for i in range(2 * str_len + 1):
            mirror = 2 * center - i
            if right > i:
                radius[i] = min(right - i, radius[mirror])
            
            while (i - 1 - radius[i] >= 0 and
                   i + 1 + radius[i] < 2 * str_len + 1 and
                   expand_str[i - 1 - radius[i]] == expand_str[i + 1 + radius[i]]):
                radius[i] += 1
            
            if i + radius[i] > right:
                center = i
                right = i + radius[i]
        return radius

    #计算两个字符串的回文半径数组
    palin_x = manacher_process(str_x)
    palin_y = manacher_process(str_y)
    longest = 0

    #遍历A中所有回文中心
    for mid in range(2 * str_len + 1):
        current_r = palin_x[mid]
        if current_r > longest:
            longest = current_r
        
        start_pos = (mid - current_r) // 2 + 1
        end_pos = (mid + current_r) // 2
        
        if 1 <= end_pos <= str_len and start_pos > 1:
            a_idx = str_len - start_pos + 2
            b_idx = end_pos
            max_possible = min(str_len - a_idx + 1, str_len - b_idx + 1)
            
            if max_possible <= 0:
                continue
            
            require = (longest - current_r) // 2 + 1
            if require > max_possible:
                continue
            
            if is_equal(a_idx, b_idx, require):
                l = require
                r = max_possible
                best = require
                while l <= r:
                    m = (l + r) >> 1
                    if is_equal(a_idx, b_idx, m):
                        best = m
                        l = m + 1
                    else:
                        r = m - 1
                if current_r + 2 * best > longest:
                    longest = current_r + 2 * best

    #遍历B中所有回文中心
    for mid in range(2 * str_len + 1):
        current_r = palin_y[mid]
        if current_r > longest:
            longest = current_r
        
        start_pos = (mid - current_r) // 2 + 1
        end_pos = (mid + current_r) // 2
        
        if 1 <= start_pos <= str_len and end_pos < str_len:
            a_idx = str_len - start_pos + 1
            b_idx = end_pos + 1
            max_possible = min(str_len - a_idx + 1, str_len - b_idx + 1)
            
            if max_possible <= 0:
                continue
            
            require = (longest - current_r) // 2 + 1
            if require > max_possible:
                continue
            
            if is_equal(a_idx, b_idx, require):
                l = require
                r = max_possible
                best = require
                while l <= r:
                    m = (l + r) >> 1
                    if is_equal(a_idx, b_idx, m):
                        best = m
                        l = m + 1
                    else:
                        r = m - 1
                if current_r + 2 * best > longest:
                    longest = current_r + 2 * best

    print(longest)

if __name__ == '__main__':
    calculate_result()

## D 优惠券

In [ ]:
## add your code here
import sys

def main():
    input_lines = sys.stdin.buffer.read().splitlines()
    ptr = 0
    output = []
    
    while ptr < len(input_lines):
        # 跳过空行
        line = input_lines[ptr].strip()
        ptr += 1
        if not line:
            continue
        
        query_count = int(line)
        # 初始化数组
        value_counter = [0] * 100005
        last_occur = [0] * 100005
        tree_arr = [0] * (query_count + 2)
        
        # 树状数组：单点更新
        def update(pos, val):
            while pos <= query_count:
                tree_arr[pos] += val
                pos += pos & -pos
        
        # 树状数组：前缀和查询
        def prefix_sum(pos):
            if pos <= 0:
                return 0
            res = 0
            while pos > 0:
                res += tree_arr[pos]
                pos -= pos & -pos
            return res
        
        # 树状数组：查找第k小
        def find_kth(k):
            pos = 0
            step = 1
            while (step << 1) <= query_count:
                step <<= 1
            while step:
                next_pos = pos + step
                if next_pos <= query_count and tree_arr[next_pos] < k:
                    pos = next_pos
                    k -= tree_arr[next_pos]
                step >>= 1
            return pos + 1
        
        total_valid = 0
        invalid_row = -1
        
        for current_row in range(1, query_count + 1):
            if invalid_row != -1:
                ptr += 1
                continue
            
            parts = input_lines[ptr].split()
            ptr += 1
            
            # 空查询操作
            if len(parts) == 1:
                update(current_row, 1)
                total_valid += 1
                continue
            
            # 插入/删除操作
            cmd = parts[0]
            num = int(parts[1])
            
            if cmd == b'I':
                value_counter[num] += 1
            else:
                value_counter[num] -= 1
            
            # 状态非法，触发撤销
            if value_counter[num] < 0 or value_counter[num] > 1:
                pre_sum = prefix_sum(last_occur[num] - 1)
                # 无可撤销元素
                if total_valid - pre_sum == 0:
                    invalid_row = current_row
                else:
                    target_pos = find_kth(pre_sum + 1)
                    update(target_pos, -1)
                    total_valid -= 1
                    
                    # 修正计数
                    if value_counter[num] < 0:
                        value_counter[num] = 0
                    else:
                        value_counter[num] = 1
            
            last_occur[num] = current_row
        
        output.append(str(invalid_row))
    
    print('\n'.join(output))

if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
## add your code here
import sys

def main():
    input_list = list(map(int, sys.stdin.read().split()))
    if not input_list:
        return
    total = input_list[0]
    point_list = []
    for idx in range(1, 2 * total, 2):
        x = input_list[idx]
        y = input_list[idx + 1]
        point_list.append((x, y))
    parent = list(range(total))
    
    def find_root(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node
    
    result = total - 1
    
    for a in range(total):
        for b in range(a + 1, total):
            x1, y1 = point_list[a]
            x2, y2 = point_list[b]
            if x1 == x2 or y1 == y2:
                root_a = find_root(a)
                root_b = find_root(b)
                if root_a != root_b:
                    parent[root_a] = root_b
                    result -= 1
    
    print(result)

if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
## add your code here
import sys

def run_matching():
    raw_input = sys.stdin.read().split()
    if not raw_input:
        return
    
    wildcard = raw_input[0]
    file_count = int(raw_input[1])
    file_list = raw_input[2: 2 + file_count]

    class FixedSegmentMatcher:
        def __init__(self, pattern):
            self.seg_len = len(pattern)
            self.fixed_chunks = []
            current_offset = 0
            for chunk in pattern.split('?'):
                if chunk:
                    self.fixed_chunks.append((chunk, current_offset))
                current_offset += len(chunk) + 1

        def is_exact_match(self, target, begin):
            if begin < 0 or begin + self.seg_len > len(target):
                return False
            for chunk, offset in self.fixed_chunks:
                start = begin + offset
                end = start + len(chunk)
                if target[start:end] != chunk:
                    return False
            return True

    class MiddleSegmentMatcher:
        def __init__(self, pattern):
            self.seg_len = len(pattern)
            self.fixed_chunks = []
            current_offset = 0
            for chunk in pattern.split('?'):
                if chunk:
                    self.fixed_chunks.append((chunk, current_offset))
                current_offset += len(chunk) + 1

        def find_position(self, target, left, right):
            if left + self.seg_len > right:
                return -1
            if not self.fixed_chunks:
                return left

            rarest_chunk = None
            best_offset = -1
            min_occur = float('inf')
            for chunk, offset in self.fixed_chunks:
                cnt = target.count(chunk, left, right)
                if cnt < min_occur:
                    min_occur = cnt
                    rarest_chunk = chunk
                    best_offset = offset

            search_begin = left + best_offset
            max_search = right - self.seg_len + best_offset + len(rarest_chunk)

            while True:
                pos = target.find(rarest_chunk, search_begin, max_search)
                if pos == -1:
                    return -1
                base = pos - best_offset
                valid = True
                for chunk, off in self.fixed_chunks:
                    if off == best_offset:
                        continue
                    st = base + off
                    ed = st + len(chunk)
                    if target[st:ed] != chunk:
                        valid = False
                        break
                if valid:
                    return base
                search_begin = pos + 1

    segments = wildcard.split('*')
    result = []

    if len(segments) == 1:
        matcher = FixedSegmentMatcher(segments[0])
        for name in file_list:
            if len(name) == matcher.seg_len and matcher.is_exact_match(name, 0):
                result.append("YES")
            else:
                result.append("NO")
        print('\n'.join(result))
        return

    prefix = segments[0]
    suffix = segments[-1]
    middle_parts = segments[1:-1]
    min_length = sum(len(s) for s in segments)

    prefix_check = FixedSegmentMatcher(prefix) if prefix else None
    suffix_check = FixedSegmentMatcher(suffix) if suffix else None
    middle_checks = [MiddleSegmentMatcher(part) for part in middle_parts if part]

    for name in file_list:
        if len(name) < min_length:
            result.append("NO")
            continue

        if prefix_check and not prefix_check.is_exact_match(name, 0):
            result.append("NO")
            continue

        suffix_start = len(name) - len(suffix)
        if suffix_check and not suffix_check.is_exact_match(name, suffix_start):
            result.append("NO")
            continue

        current = len(prefix)
        end_bound = len(name) - len(suffix)
        ok = True

        for matcher in middle_checks:
            found_at = matcher.find_position(name, current, end_bound)
            if found_at == -1:
                ok = False
                break
            current = found_at + matcher.seg_len

        result.append("YES" if ok else "NO")

    print('\n'.join(result))

if __name__ == "__main__":
    run_matching()

## G 汉诺塔

In [ ]:
## add your code here
import sys

def compute_hanoi_steps():
    try:
        first_line = sys.stdin.readline()
        if not first_line:
            return
        disk_count = int(first_line.strip())
        rule_list = sys.stdin.readline().split()
    except EOFError:
        return

    # 柱子编号映射 A->0, B->1, C->2
    peg_map = {'A': 0, 'B': 1, 'C': 2}
    
    # dp[m][p]：把 m 个盘子从柱子 p 完全移走需要的总步数
    dp = [[0] * 3 for _ in range(disk_count + 1)]
    # dest[m][p]：把 m 个盘子从柱子 p 移走后最终停在的目标柱子
    dest = [[0] * 3 for _ in range(disk_count + 1)]

    for src in range(3):
        for rule in rule_list:
            s = peg_map[rule[0]]
            e = peg_map[rule[1]]
            if s == src:
                dp[1][src] = 1
                dest[1][src] = e
                break

    for m in range(2, disk_count + 1):
        for src_peg in range(3):
            mid_peg = dest[m-1][src_peg]
            spare_peg = 3 - src_peg - mid_peg

            if dest[m-1][mid_peg] != src_peg:
                dp[m][src_peg] = dp[m-1][src_peg] + 1 + dp[m-1][mid_peg]
                dest[m][src_peg] = dest[m-1][mid_peg]
            else:
                dp[m][src_peg] = dp[m-1][src_peg] + 1 + dp[m-1][mid_peg] + 1 + dp[m-1][src_peg]
                dest[m][src_peg] = mid_peg

    # 输出从柱子 A(0) 移走 n 个盘子的总步数
    print(dp[disk_count][0])

if __name__ == "__main__":
    compute_hanoi_steps()

## H 马步距离

In [ ]:
## add your code here
import sys

def calculate_min_steps():
    raw_input = sys.stdin.read().split()
    if not raw_input:
        return
    
    # 读取起点和终点坐标
    start_x, start_y, end_x, end_y = map(int, raw_input)
    
    # 计算绝对距离并转为非负数
    delta_x = abs(end_x - start_x)
    delta_y = abs(end_y - start_y)
    
    # 统一格式：保证 dx >= dy
    dx, dy = delta_x, delta_y
    if dx < dy:
        dx, dy = dy, dx
    
    # 特殊边界情况直接返回结果
    if dx == 0 and dy == 0:
        print(0)
        return
    if dx == 1 and dy == 0:
        print(3)
        return
    if dx == 1 and dy == 1:
        print(2)
        return
    if dx == 2 and dy == 2:
        print(4)
        return
    
    # 核心数学公式计算基础步数
    base_step1 = (dx + 1) // 2
    base_step2 = (dx + dy + 2) // 3
    result = max(base_step1, base_step2)
    
    # 奇偶校验修正
    if (result % 2) != ((dx + dy) % 2):
        result += 1
    
    print(result)
if __name__ == "__main__":
    calculate_min_steps()

## I 直方图最大矩形

In [ ]:
## add your code here
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        bars = [0] + heights + [0]
        stack = [0]
        max_area = 0
        
        for right_idx in range(1, len(bars)):
            while bars[right_idx] < bars[stack[-1]]:
                # 计算以当前高度为高的矩形面积
                curr = stack.pop()
                left_idx = stack[-1]
                width = right_idx - left_idx - 1
                current_area = bars[curr] * width
                
                # 更新最大面积
                if current_area > max_area:
                    max_area = current_area
            
            stack.append(right_idx)
        
        return max_area

## J 消防局的设立

In [ ]:
## add your code here
import sys

def calculate_fire_stations():
    data = sys.stdin.read().split()
    if not data:
        return
    
    node_count = int(data[0])
    if node_count == 0:
        print(0)
        return
    
    father = [0] * (node_count + 1)
    for idx in range(2, node_count + 1):
        father[idx] = int(data[idx - 1])
    
    far_uncover = [0] * (node_count + 1)
    near_fire = [float('inf')] * (node_count + 1)
    
    total_stations = 0

    for current in range(node_count, 0, -1):
        if far_uncover[current] + near_fire[current] <= 2:
            far_uncover[current] = -float('inf')
        
        # 未覆盖点距离为2，必须建站
        if far_uncover[current] == 2:
            total_stations += 1
            near_fire[current] = 0
            far_uncover[current] = -float('inf')
        
        # 非根节点，向上传递信息
        if current != 1:
            p_node = father[current]
            far_uncover[p_node] = max(far_uncover[p_node], far_uncover[current] + 1)
            near_fire[p_node] = min(near_fire[p_node], near_fire[current] + 1)
    
    # 根节点仍有未覆盖点，需要补一个站
    if far_uncover[1] >= 0:
        total_stations += 1
    
    print(total_stations)

if __name__ == "__main__":
    calculate_fire_stations()